In [1]:
%cd /home/parthgandhi/Projects/SwingBot/swingbot/src

/home/parthgandhi/Projects/SwingBot/swingbot/src


In [2]:
import logging
from datetime import datetime, timedelta

import polars as pl

from config.ingestion.brokers import KiteConfig
from config.ingestion.data_sources import NSEConfig
from ingestion import fetch_nse_indices
from utils import setup_logger
from computation.scanner import prep_scan_data, find_stocks, basic_scan
from computation.filter import basic_filter, adr_filter, pullback_filter

logger = logging.getLogger(__name__)

setup_logger()

In [3]:
end_date = "2026-03-13"
SCAN_DATE = datetime(2026, 3, 13)
start_date = SCAN_DATE - timedelta(days=KiteConfig.HIST_DATA_START_LOOKBACK * 30)

In [4]:
nse_indices_df = fetch_nse_indices(download_flag=False)

# get industry classification data
nse_ind_table_id = NSEConfig.get_db_tbl(
    tbl_name=NSEConfig.CLASSIFICATION_TABLE_ID, failed_tbl=False
)

query = f"""
select max(timestamp) from {nse_ind_table_id}
"""
max_timestamp = pl.read_database_uri(query=query, uri=NSEConfig.DB_CONN).item(0, 0)

logger.info(f"TimeStamp usef for NSE Industry Classification: {max_timestamp}")

query = f"""
select * from {nse_ind_table_id}
where timestamp = '{max_timestamp}'
"""

nse_ind_df = pl.read_database_uri(query=query, uri=NSEConfig.DB_CONN)

# Get Stock & Indices Data
stock_table_id = KiteConfig.get_db_tbl(
    data_type=KiteConfig.TYP_STOCKS, frequency="day", failed_tbl=False
)
indices_table_id = KiteConfig.get_db_tbl(
    data_type=KiteConfig.TYP_INDICES, frequency="day", failed_tbl=False
)

query = f"""
    select *
    from {stock_table_id}
"""

stocks_df = pl.read_database_uri(query=query, uri=KiteConfig.DB_CONN)

query = f"""
    select *
    from {indices_table_id}
"""

indices_df = pl.read_database_uri(query=query, uri=KiteConfig.DB_CONN)

2026-03-15 11:05:45 | ERROR | ingestion.ingest | fetch_nse_indices | Failed for /home/parthgandhi/data/tmp/nse/indices/nse_index_nifty_reits_realty.csv. Error: found more fields than defined in 'Schema'

Consider setting 'truncate_ragged_lines=True'.
2026-03-15 11:05:45 | INFO | ingestion.ingest | fetch_nse_indices | Fetched Indices Data for 29/30
2026-03-15 11:05:45 | INFO | __main__ | <module> | TimeStamp usef for NSE Industry Classification: 2026-03-14


In [5]:
scan_df = prep_scan_data(data=stocks_df)
basic_scan_df = basic_scan(data=scan_df)
find_stocks_scan_df = find_stocks(
    data=basic_scan_df, start_date=start_date, end_date=SCAN_DATE
).collect()
scan_stocks_list = find_stocks_scan_df.get_column("symbol").to_list()

2026-03-15 11:05:46 | INFO | computation.scanner | find_stocks | MIN DATE IN DATA: 2026-01-07 & START DATE is 2025-12-13 00:00:00
2026-03-15 11:05:46 | INFO | computation.scanner | find_stocks | MAX DATE IN DATA: 2026-03-13 & END DATE IS 2026-03-13 00:00:00


In [6]:
basic_filter_df = basic_filter(
    data=scan_df, symbol_list=scan_stocks_list, scan_date=SCAN_DATE
)
basic_filter_stocks = basic_filter_df.get_column("symbol").to_list()

2026-03-15 11:05:46 | INFO | computation.filter | basic_filter | Applying Basic Filter
2026-03-15 11:05:46 | INFO | computation.filter | basic_filter | Number of stocks in symbol list: 1717
2026-03-15 11:05:46 | INFO | computation.filter | basic_filter | Symbols after basic filter: 422


In [7]:
adr_filter_df = adr_filter(
    data=scan_df, symbol_list=basic_filter_stocks, scan_date=SCAN_DATE, adr_cutoff=3.5
)
adr_filter_stocks = adr_filter_df.get_column("symbol").to_list()

2026-03-15 11:05:46 | INFO | computation.filter | adr_filter | Applying ADR Filter
2026-03-15 11:05:46 | INFO | computation.filter | adr_filter | Number of stocks in symbol list: 422
2026-03-15 11:05:47 | INFO | computation.filter | adr_filter | Symbols after basic filter: 291


In [10]:
pullback_filter_df = pullback_filter(
    data=scan_df, symbol_list=basic_filter_stocks, scan_date=SCAN_DATE
)
pullback_filter_stocks = pullback_filter_df.get_column("symbol").unique().to_list()

2026-03-15 11:06:09 | INFO | computation.filter | pullback_filter | Applying PullBack Filter
2026-03-15 11:06:09 | INFO | computation.filter | pullback_filter | Number of stocks in symbol list: 422
2026-03-15 11:06:09 | INFO | computation.filter | pullback_filter | Symbols after ADR filter: 318


In [27]:
final_res = (
    scan_df.filter(
        (pl.col("symbol").is_in(basic_filter_stocks))
        & (pl.col("timestamp") == SCAN_DATE)
    )
    .with_columns(
        pl.when(pl.col("symbol").is_in(value)).then(True).otherwise(False).alias(key)
        for key, value in {
            "basic_filter_flag": basic_filter_stocks,
            "adr_filter_flag": adr_filter_stocks,
            "pullback_filter_flag": pullback_filter_stocks,
        }.items()
    )
    .join(
        pullback_filter_df.lazy().select(
            "symbol",
            "mid_down_streak",
            "near_ema_9",
            "near_ema_21",
            "near_sma_50",
        ),
        on="symbol",
        how="left",
    )
    .join(nse_ind_df.lazy(), on="symbol", how="left")
)

final_res.collect()

symbol,timestamp,open,high,low,close,volume,close_sma_50,close_sma_200,close_ema_9,close_ema_21,volume_sma_20,volume_sma_50,day_range,body_by_range_pct_sma_20,body_by_range_pct_sma_50,lower_wick_by_range_pct_sma_20,lower_wick_by_range_pct_sma_50,upper_wick_by_range_pct_sma_20,upper_wick_by_range_pct_sma_50,adr_pct_20,rvol_pct_20,rvol_pct_50,clean_score_pct_20,clean_score_pct_50,close_prev_1,close_prev_5,close_prev_21,close_prev_63,close_prev_126,timestamp_prev_1,timestamp_prev_5,timestamp_prev_21,timestamp_prev_63,timestamp_prev_126,pct_gain_prev_1,pct_gain_prev_5,pct_gain_prev_21,pct_gain_prev_63,pct_gain_prev_126,all_data_flag,basic_filter_flag,adr_filter_flag,pullback_filter_flag,mid_down_streak,near_ema_9,near_ema_21,near_sma_50,timestamp_right,macro_economic_sector,sector,industry,basic_industry,market_cap_cr
str,date,f64,f64,f64,f64,i64,f64,f64,f64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,date,date,date,date,date,f64,f64,f64,f64,f64,bool,bool,bool,bool,u32,bool,bool,bool,str,str,str,str,str,f64
"""AARTIIND""",2026-03-13,447.95,447.95,419.25,420.75,1148207,407.9,405.79,429.89,430.92,839815,1144439,1.0685,46.85,43.35,27.64,33.29,25.51,23.35,3.96,137.0,100.0,33.9,28.92,446.2,419.35,467.15,357.0,388.9,2026-03-12,2026-03-06,2026-02-11,2025-12-11,2025-09-10,-5.7037,0.3339,-9.9326,17.8571,8.1898,true,true,true,true,1,true,true,false,"""2026-03-14""","""Commodities""","""Chemicals""","""Chemicals & Petrochemicals""","""Specialty Chemicals""",15228.96
"""ABB""",2026-03-13,6439.5,6554.0,6339.5,6392.5,1070957,5565.48,5430.92,6193.65,6014.23,592895,430933,1.0338,51.41,45.54,26.35,29.59,22.24,24.87,3.32,181.0,249.0,37.86,32.06,6409.0,6062.0,5825.5,5242.5,5161.9,2026-03-12,2026-03-06,2026-02-11,2025-12-11,2025-09-10,-0.2575,5.452,9.7331,21.9361,23.8401,true,true,false,false,null,null,null,null,"""2026-03-14""","""Industrials""","""Capital Goods""","""Electrical Equipment""","""Heavy Electrical Equipment""",135388.26
"""STYRENIX""",2026-03-13,2078.0,2105.0,1973.2,2031.5,67507,1937.73,2461.38,1974.55,1951.32,37835,30516,1.0668,39.22,38.94,34.28,31.42,26.49,29.65,2.84,178.0,221.0,25.78,26.71,2073.8,1864.7,1989.6,2055.1,2647.8,2026-03-12,2026-03-06,2026-02-11,2025-12-11,2025-09-10,-2.0397,8.9451,2.106,-1.1484,-23.2759,true,true,false,true,1,true,true,true,null,null,null,null,null,null
"""GODREJAGRO""",2026-03-13,583.0,586.0,561.25,568.95,146925,579.38,674.35,596.54,601.47,228574,337835,1.0441,45.16,48.32,32.07,28.26,22.77,23.43,3.34,64.0,43.0,30.68,34.66,586.55,595.2,591.3,591.65,740.6,2026-03-12,2026-03-06,2026-02-11,2025-12-11,2025-09-10,-3.0006,-4.4103,-3.7798,-3.8367,-23.1772,true,true,false,true,3,false,false,true,null,null,null,null,null,null
"""APCOTEXIND""",2026-03-13,358.3,363.45,344.05,352.9,99953,363.69,383.29,361.9,364.82,23576,22211,1.0564,31.54,38.87,37.27,32.89,31.19,28.24,2.83,424.0,450.0,19.79,26.09,358.2,367.6,370.75,368.5,409.35,2026-03-12,2026-03-06,2026-02-11,2025-12-11,2025-09-10,-1.4796,-3.9989,-4.8146,-4.2334,-13.7902,true,true,false,true,2,true,false,false,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ATHERENERG""",2026-03-13,710.0,721.0,692.1,705.0,2014478,679.85,555.48,699.94,698.08,1468082,1655575,1.0418,35.62,40.15,42.6,37.9,21.78,21.96,3.99,137.0,122.0,20.45,24.93,710.55,674.05,716.25,646.45,541.55,2026-03-12,2026-03-06,2026-02-11,2025-12-11,2025-09-10,-0.7811,4.5916,-1.5707,9.0572,30.1819,true,true,true,true,0,true,true,true,null,null,null,null,null,null
"""PASHUPATI""",2026-03-13,1014.8,1014.8,945.0,988.45,26637,875.38,763.04,992.63,961.56,6304,5656,1.0739,36.86,39.39,45.38,39.73,17.76,20.89,4.95,423.0,471.0,20.13,23.74,996.1,1011.95,864.4,831.05,672.5,2026-03-12,2026-03-06,2026-02-11,2025-12-11,2025-09-10,-0.768,-2.3222,14.351,18.9399,46.9814,true,true,true,true,1,true,true,false,null,null,null,null,null,null
"""KNAGRI""",2026-03-13,185.0,185.0,173.15,178.8,24136,177.98,211.